# Reddit Web3 爬蟲教學

這份 Notebook 示範如何用 Python 爬取 Reddit 公開 JSON endpoint，收集 Web3 相關 subreddit 的貼文與留言，整理成 CSV，最後交給本專案的 Web3 Knowledge Graph pipeline 做 entity/topic 圖譜分析。

本教學使用的是公開頁面的 JSON endpoint，不需要 Reddit API key，適合課堂展示與小量資料實作。請保持低頻率請求、設定清楚的 User-Agent，並避免收集私人或敏感資料。

## 1. 爬蟲流程總覽

1. 指定 subreddit，例如 `ethereum`、`defi`、`CryptoCurrency`。
2. 用 `https://www.reddit.com/r/{subreddit}/hot.json` 取得貼文列表。
3. 用 `after` 參數做分頁。
4. 解析貼文欄位：標題、內文、日期、分數、留言數、連結。
5. 用 `https://www.reddit.com/comments/{post_id}.json` 取得留言。
6. 過濾 stickied、空白、`[deleted]`、`[removed]` 等雜訊。
7. 輸出成 `reddit_web3_posts_from_notebook.csv`。
8. 執行 Web3 Knowledge Graph pipeline。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import time

import pandas as pd
import requests

## 2. 設定爬蟲參數

`POSTS_PER_SUBREDDIT` 控制每個 subreddit 抓幾篇貼文；`COMMENTS_PER_POST` 控制每篇貼文最多抓幾則留言。課堂展示建議先從小量開始，避免請求太密集。

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "docs":
    PROJECT_ROOT = PROJECT_ROOT.parent

SUBREDDITS = ["ethereum", "defi", "CryptoCurrency", "solana", "web3", "NFT"]
SORT = "hot"
POSTS_PER_SUBREDDIT = 10
COMMENTS_PER_POST = 3
COMMENT_DEPTH = 2
COMMENT_SORT = "top"
PAGE_LIMIT = 25
SLEEP_SECONDS = 1.0
USER_AGENT = "web3-kg-coursework-notebook/1.0 by zoo100130"
OUTPUT_PATH = PROJECT_ROOT / "data" / "reddit_web3_posts_from_notebook.csv"

OUTPUT_PATH

## 3. 建立共用請求函式

Reddit 對沒有 User-Agent 或請求太頻繁的程式比較容易拒絕，所以每次請求都會帶上 User-Agent，並在爬取時加入 `sleep`。

In [ ]:
def fetch_json(url, params=None, timeout=20):
    response = requests.get(
        url,
        params=params or {},
        headers={"User-Agent": USER_AGENT},
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()

## 4. 抓取 subreddit 貼文列表

Reddit listing endpoint 會回傳 `children` 與 `after`。`children` 是本頁貼文；`after` 是下一頁游標。

In [ ]:
def fetch_listing(subreddit, sort="hot", limit=25, after=None):
    url = f"https://www.reddit.com/r/{subreddit}/{sort}.json"
    params = {"limit": min(limit, 100)}
    if after:
        params["after"] = after
    payload = fetch_json(url, params=params)
    return payload if isinstance(payload, dict) else {}

## 5. 解析貼文欄位

Knowledge Graph pipeline 需要 `post_id`、`subreddit`、`title`、`selftext`、`created_utc`、`score`、`url`。這裡額外加入 `content_type`，讓貼文和留言可以在同一份 CSV 裡區分。

In [ ]:
def format_reddit_date(created_utc):
    if not created_utc:
        return ""
    return datetime.fromtimestamp(float(created_utc), tz=timezone.utc).date().isoformat()


def parse_post(child):
    data = child.get("data", {})
    return {
        "post_id": data.get("id", ""),
        "content_type": "post",
        "parent_post_id": "",
        "comment_id": "",
        "subreddit": data.get("subreddit", ""),
        "title": data.get("title", ""),
        "selftext": data.get("selftext", "") or "",
        "created_utc": format_reddit_date(data.get("created_utc")),
        "score": data.get("score", 0),
        "url": data.get("url", ""),
        "permalink": "https://www.reddit.com" + data.get("permalink", ""),
        "num_comments": data.get("num_comments", 0),
        "author": data.get("author", ""),
        "upvote_ratio": data.get("upvote_ratio", ""),
        "stickied": bool(data.get("stickied", False)),
    }

## 6. 過濾常見雜訊貼文

許多 subreddit 會有置頂的 daily discussion 或 rules 貼文。這些通常不是我們要分析的 Web3 新聞/討論，因此先排除。

In [ ]:
def is_noise_post(row):
    title = str(row.get("title", "")).lower()
    noise_terms = [
        "daily discussion",
        "general discussion",
        "weekly discussion",
        "megathread",
        "welcome to",
        "subreddit rules",
    ]
    return row.get("stickied") or any(term in title for term in noise_terms)

## 7. 抓取留言 endpoint

留言 endpoint 的網址是 `https://www.reddit.com/comments/{post_id}.json`。回傳通常是一個 list：第 1 個元素是原貼文，第 2 個元素是留言樹。

In [ ]:
def fetch_comments(post_id, limit=10, depth=2, sort="top"):
    url = f"https://www.reddit.com/comments/{post_id}.json"
    params = {
        "limit": min(max(limit, 1), 500),
        "depth": max(depth, 1),
        "sort": sort,
    }
    payload = fetch_json(url, params=params)
    return payload if isinstance(payload, list) else []

## 8. 解析留言資料

留言會被存成 `content_type=comment`，並用 `parent_post_id` 保留它來自哪一篇貼文。為了讓 pipeline 能直接分析，留言本文放在 `selftext` 欄位。

In [ ]:
def parse_comment(child, parent_post):
    data = child.get("data", {})
    comment_id = data.get("id", "")
    parent_title = str(parent_post.get("title", ""))[:120]
    return {
        "post_id": comment_id,
        "content_type": "comment",
        "parent_post_id": parent_post.get("post_id", ""),
        "comment_id": comment_id,
        "subreddit": data.get("subreddit", parent_post.get("subreddit", "")),
        "title": f"Comment on: {parent_title}",
        "selftext": data.get("body", "") or "",
        "created_utc": format_reddit_date(data.get("created_utc")),
        "score": data.get("score", 0),
        "url": parent_post.get("url", ""),
        "permalink": "https://www.reddit.com" + data.get("permalink", ""),
        "num_comments": 0,
        "author": data.get("author", ""),
        "upvote_ratio": "",
        "stickied": bool(data.get("stickied", False)),
    }


def is_noise_comment(row):
    body = str(row.get("selftext", "")).strip().lower()
    return not body or body in {"[deleted]", "[removed]"} or row.get("stickied")

## 9. 攤平留言樹

Reddit 留言是樹狀結構，一則留言下面可能還有回覆。這個函式會用遞迴把 `t1` comment 節點攤平成一串留言。

In [ ]:
def iter_comment_children(children):
    for child in children:
        if child.get("kind") != "t1":
            continue
        yield child
        replies = child.get("data", {}).get("replies")
        if isinstance(replies, dict):
            nested = replies.get("data", {}).get("children", [])
            yield from iter_comment_children(nested)

## 10. 抓取單篇貼文的留言

`COMMENTS_PER_POST` 不建議一開始設太大。Web3 主題分析通常先抓每篇熱門留言 3 到 5 則，就能看到討論方向。

In [ ]:
def crawl_comments_for_post(parent_post, comments_per_post=3, depth=2, sort="top"):
    if comments_per_post <= 0:
        return []

    payload = fetch_comments(
        parent_post["post_id"],
        limit=max(comments_per_post * 3, 10),
        depth=depth,
        sort=sort,
    )
    if len(payload) < 2 or not isinstance(payload[1], dict):
        return []

    rows = []
    children = payload[1].get("data", {}).get("children", [])
    for child in iter_comment_children(children):
        row = parse_comment(child, parent_post)
        if not is_noise_comment(row):
            rows.append(row)
        if len(rows) >= comments_per_post:
            break
    return rows

## 11. 爬取單一 subreddit

先抓貼文，再依序補抓每篇貼文的留言。每次請求之間都休息一下，避免短時間打太多請求。

In [ ]:
def crawl_subreddit(subreddit):
    rows = []
    post_rows = []
    after = None

    while len(post_rows) < POSTS_PER_SUBREDDIT:
        payload = fetch_listing(subreddit, sort=SORT, limit=PAGE_LIMIT, after=after)
        listing = payload.get("data", {})
        children = listing.get("children", [])
        if not children:
            break

        for child in children:
            if child.get("kind") != "t3":
                continue
            row = parse_post(child)
            if not is_noise_post(row):
                post_rows.append(row)
            if len(post_rows) >= POSTS_PER_SUBREDDIT:
                break

        after = listing.get("after")
        if not after:
            break
        time.sleep(SLEEP_SECONDS)

    for post in post_rows:
        rows.append(post)
        comments = crawl_comments_for_post(
            post,
            comments_per_post=COMMENTS_PER_POST,
            depth=COMMENT_DEPTH,
            sort=COMMENT_SORT,
        )
        rows.extend(comments)
        time.sleep(SLEEP_SECONDS)

    return rows

## 12. 爬取多個 subreddit

這裡會把所有 subreddit 的結果合併成同一個 DataFrame。若某個 subreddit 暫時拒絕請求，程式會跳過它，不讓整個流程中斷。

In [ ]:
all_rows = []
for subreddit in SUBREDDITS:
    print(f"Crawling r/{subreddit}...")
    try:
        rows = crawl_subreddit(subreddit)
        post_count = sum(row["content_type"] == "post" for row in rows)
        comment_count = sum(row["content_type"] == "comment" for row in rows)
        print(f"  fetched {post_count} posts and {comment_count} comments")
        all_rows.extend(rows)
    except requests.HTTPError as exc:
        print(f"  skipped r/{subreddit}: HTTP {exc.response.status_code}")
    except requests.RequestException as exc:
        print(f"  skipped r/{subreddit}: {exc}")
    time.sleep(SLEEP_SECONDS)

posts_df = pd.DataFrame(all_rows)
posts_df.head()

## 13. 檢查資料分布

用 `content_type` 可以快速確認抓到多少貼文與留言；用 subreddit 分組可以確認資料是否平均。

In [ ]:
if posts_df.empty:
    print("No rows crawled")
else:
    display(posts_df.groupby("content_type").size())
    display(posts_df.groupby(["subreddit", "content_type"]).size())

## 14. 清理欄位並輸出 CSV

這份 CSV 可以直接交給本專案的 `src/pipeline.py` 使用。`utf-8-sig` 可以讓 Excel 在 Windows 上比較容易正確辨識中文與英文內容。

In [ ]:
OUTPUT_COLUMNS = [
    "post_id",
    "content_type",
    "parent_post_id",
    "comment_id",
    "subreddit",
    "title",
    "selftext",
    "created_utc",
    "score",
    "url",
    "permalink",
    "num_comments",
    "author",
    "upvote_ratio",
    "stickied",
]

for column in OUTPUT_COLUMNS:
    if column not in posts_df.columns:
        posts_df[column] = ""

posts_df = posts_df[OUTPUT_COLUMNS].drop_duplicates(subset=["content_type", "post_id"])
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
posts_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
OUTPUT_PATH

## 15. 用爬到的資料建立 Web3 Knowledge Graph

回到 PowerShell，在 `web3_kg_project` 資料夾執行：

```powershell
python .\src\pipeline.py --input .\dataeddit_web3_posts_from_notebook.csv --output .\outputseddit_live_from_notebook
python -m http.server 8080
```

然後打開：

```text
http://localhost:8080/outputs/reddit_live_from_notebook/interactive_graph.html
```

In [ ]:
# 如果你想在 Notebook 裡直接執行 pipeline，可以取消下面註解。
# import subprocess
# subprocess.run([
#     "python", str(PROJECT_ROOT / "src" / "pipeline.py"),
#     "--input", str(OUTPUT_PATH),
#     "--output", str(PROJECT_ROOT / "outputs" / "reddit_live_from_notebook"),
# ], check=True)

## 16. 常見問題

- 被 Reddit 拒絕請求：把 `SLEEP_SECONDS` 調成 3 或更高，並降低 `POSTS_PER_SUBREDDIT` 與 `COMMENTS_PER_POST`。
- 抓不到留言：有些貼文留言很少，或留言被刪除，這是正常現象。
- 中文終端顯示亂碼：通常是 PowerShell code page 問題，CSV 與 Notebook 本身仍是 UTF-8。
- 想正式大量收集資料：建議改用 Reddit 官方 API 或 PRAW，並使用 OAuth。

## 17. 延伸練習

1. 把 `SORT` 改成 `new` 或 `top`，比較熱門貼文與最新貼文的差異。
2. 增加 `COMMENTS_PER_POST`，觀察留言是否讓 entity co-mention network 更密集。
3. 加入情緒分析欄位，分析不同 chain / protocol 的討論情緒。
4. 把 `parent_post_id` 做成圖譜關係，建立 Post -> Comment 討論串圖。
5. 把輸出的 `neo4j_import.cypher` 匯入 Neo4j，用 Cypher 查詢 Web3 社群主題。